In [1]:
# =========================================
# ILLINOIS AGILITY TEST - FULL BIO PIPELINE
# =========================================

import cv2
import numpy as np
import mediapipe as mp
from scipy.signal import find_peaks
import math
import csv
from ultralytics import YOLO

In [2]:
# -------------------------------
# MediaPipe Setup
# -------------------------------
mpPose = mp.solutions.pose
pose = mpPose.Pose(min_detection_confidence=0.6)

In [3]:
def get_display_size(frame_width, frame_height, margin=100):
    try:
        user32 = ctypes.windll.user32
        screen_width = user32.GetSystemMetrics(0)
        screen_height = user32.GetSystemMetrics(1)
    except Exception:
        screen_width, screen_height = frame_width, frame_height

    max_width = max(screen_width - margin, 1)
    max_height = max(screen_height - margin, 1)
    scale = min(max_width / max(frame_width, 1), max_height / max(frame_height, 1))

    return max(1, int(frame_width * scale)), max(1, int(frame_height * scale))

In [4]:
# -------------------------------
# Utility Functions
# -------------------------------
def safe_mean(arr, default=0.0):
    return sum(arr)/len(arr) if len(arr) > 0 else default

def safe_std(arr, default=0.0):
    return float(np.std(arr)) if len(arr) > 0 else default

In [5]:
# -------------------------------
# Cone Detection
# -------------------------------
def detect_cones(path):
    try:
        model = YOLO("best.pt")
    except:
        print("c1")
        return None

    cap = cv2.VideoCapture(path)
    centers = [[] for _ in range(5)]
    boxes = []
    

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame, conf=0.5, verbose=False)

        if len(results[0].boxes) > 4:
            boxes = []
            for box in results[0].boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = box.conf.item()
                cx = (x1 + x2) // 2
                cy = (y1 + y2) // 2
                # boxes.append((cx, cy))
                boxes.append(cx)

                # Draw rectangle on cone
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

                # Optional: confidence label
                cv2.putText(frame, f"Cone {conf:.2f}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            
            boxes.sort()
            for i in range(5):
                centers[i].append(boxes[i])

            boxes.clear()
        if len(centers[0]) > 50:
            break
            
            
        cv2.namedWindow("Video", cv2.WINDOW_NORMAL)
        cv2.setWindowProperty("Video", cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)
        cv2.imshow("Video", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

    mean_centers = []

    for i in range(5):
        if len(centers[i]) > 0:
            # print(f"cordinate-> {centers[i]}")
            mean_centers.append(int(np.mean(centers[i][0])))
    
    return mean_centers

In [6]:
path = "data/agility_test_1.mp4"
points = detect_cones(path)
print(points)

[20, 236, 464, 646, 831]


In [7]:
def fallback_cones(w, h):
    return [
        (int(w*0.2), int(h*0.5)),
        (int(w*0.8), int(h*0.5)),
        (int(w*0.4), int(h*0.4)),
        (int(w*0.6), int(h*0.6))
    ]

In [8]:
# -------------------------------
# NEW: TORSO ROTATION
# -------------------------------
def torso_angle(lms):
    l_sh, r_sh = lms[11], lms[12]
    l_hip, r_hip = lms[23], lms[24]

    sh_vec = (r_sh.x - l_sh.x, r_sh.y - l_sh.y)
    hip_vec = (r_hip.x - l_hip.x, r_hip.y - l_hip.y)

    dot = sh_vec[0]*hip_vec[0] + sh_vec[1]*hip_vec[1]
    mag1 = math.sqrt(sh_vec[0]**2 + sh_vec[1]**2)
    mag2 = math.sqrt(hip_vec[0]**2 + hip_vec[1]**2)

    if mag1 == 0 or mag2 == 0:
        return 0

    return math.degrees(math.acos(dot/(mag1*mag2)))

In [9]:
# -------------------------------
# UPDATED METRICS
# -------------------------------
def compute_metrics(hip_x, hip_y, left_y, right_y, time_list):

    hip_x = np.array(hip_x)
    hip_y = np.array(hip_y)
    left_y = np.array(left_y)
    right_y = np.array(right_y)
    t = np.array(time_list)

    # -------- RHYTHM (FEET BASED) --------
    peaks_l, _ = find_peaks(-left_y, distance=5)
    peaks_r, _ = find_peaks(-right_y, distance=5)

    if len(peaks_l) > 1:
        intervals_l = np.diff(t[peaks_l])
    else:
        intervals_l = [0.2]

    if len(peaks_r) > 1:
        intervals_r = np.diff(t[peaks_r])
    else:
        intervals_r = [0.2]

    rhythm = safe_std(list(intervals_l) + list(intervals_r), 0.2)

    # -------- LIMB SYNCHRONIZATION --------
    min_len = min(len(left_y), len(right_y))
    sync_diff = safe_mean(np.abs(left_y[:min_len] - right_y[:min_len]))

    # -------- BALANCE --------
    sway = safe_std(hip_x)

    # -------- FLUIDITY --------
    if len(t) > 3:
        vel = np.diff(hip_y) / np.diff(t)
        accel = np.diff(vel)
        jerk = np.diff(accel)
        accel_var = safe_std(accel, 1.0)
        jerk_var = safe_std(jerk, 1.0)
    else:
        accel_var = 1.0
        jerk_var = 1.0

    return rhythm, sync_diff, sway, accel_var, jerk_var

In [10]:
# -------------------------------
# NEW: FOOT-BASED CONE ACCURACY
# -------------------------------
def cone_accuracy_feet(touch_point, cones, cone_hits):
    # print(cones)
    cone_number = -1
    for cx in cones:
        cone_number += 1
        if touch_point >= cx:
            cone_hits.add(cone_number)

    return cone_hits

In [11]:
# -------------------------------
# SCORING FUNCTIONS
# -------------------------------
def time_score(t):
    if t <= 15.2: return 5
    elif t <= 16.8: return 4
    elif t <= 18.4: return 3
    elif t <= 20.0: return 2
    else: return 1

In [12]:
def balance_score(sway):
    if sway < 0.02: return 2
    elif sway < 0.05: return 1
    else: return 0

In [13]:
def rhythm_score(rhythm):
    if rhythm < 0.05: return 2
    elif rhythm < 0.08: return 1
    else: return 0

In [14]:
def sync_score(sync_diff):
    if sync_diff < 0.02: return 2
    elif sync_diff < 0.05: return 1
    else: return 0

In [15]:
def accuracy_score(cone_hits):
    hits = len(cone_hits)-1
    if hits >= 4: return 2
    elif hits >= 2: return 1
    else: return 0

In [16]:
def fluidity_score(accel_var, jerk_var):
    if accel_var < 0.04 and jerk_var < 0.1: return 2
    elif accel_var < 0.08: return 1
    else: return 0

In [17]:
def turning_score(angle):
    if angle < 20: return 2
    elif angle < 30: return 1
    else: return 0

In [18]:
# -------------------------------
# MAIN FUNCTION
# -------------------------------
def illinois_agility(path):

    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_width = int(cap.get(3))
    frame_height = int(cap.get(4))
    display_width, display_height = get_display_size(frame_width, frame_height)

    cones = detect_cones(path)
    if cones is None:
        cones = fallback_cones(frame_width, frame_height)

    hip_x, hip_y, time_list = [], [], []
    left_y, right_y = [], []
    foot_positions = []
    torso_angles = []

    frame_idx = 0
    window_name = "Agility Test Processing"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window_name, display_width, display_height)

    cone_hits = set()
    zero_cone_flag = True

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(rgb)
        for centre in cones:
            cv2.circle(frame, (centre, int(0.5*frame_height)), radius=10, color=(0, 255, 0), thickness=-1)
        
        if results.pose_landmarks:
            lms = results.pose_landmarks.landmark

            # -------- HIP --------
            hx = (lms[23].x + lms[24].x) / 2
            hy = (lms[23].y + lms[24].y) / 2

            hip_x.append(hx)
            hip_y.append(hy)
            time_list.append(frame_idx / fps if fps else 0)

            # -------- FEET --------
            lf_x = int(lms[31].x * frame_width)
            lf_y = int(lms[31].y * frame_height)
            rf_x = int(lms[32].x * frame_width)
            rf_y = int(lms[32].y * frame_height)

            left_y.append(lms[31].y)
            right_y.append(lms[32].y)

            foot_positions.append((lf_x, lf_y))
            foot_positions.append((rf_x, rf_y))

            # -------- HAND --------
            # lh_x = int(lms[19].x * frame_width)
            # rh_y = int(lms[20].y * frame_height)

            hip_centre = (int(frame_width*(lms[23].x + lms[24].x)/2), int(frame_height*(lms[23].y + lms[24].y)/2))
            cv2.circle(frame, hip_centre, radius=10, color=(0, 255, 0), thickness=-1)
            touch_point = (frame_width*(lms[23].x + lms[24].x)/2)
            if touch_point < cones[0]:
                zero_cone_flag = True
            if zero_cone_flag:
                before_len = len(cone_hits)
                cone_hits = cone_accuracy_feet(touch_point, cones, cone_hits)
                if len(cone_hits) > 1 and len(cone_hits) == before_len + 1:
                    zero_cone_flag = False
                
            cv2.putText(frame, f"Cone Hits-> {cone_hits}, {zero_cone_flag}", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2, cv2.LINE_AA)

            # -------- TORSO --------
            torso_angles.append(torso_angle(lms))

        cv2.imshow(window_name, frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

        frame_idx += 1

    cap.release()
    cv2.destroyAllWindows()

    total_time = frame_idx / fps if fps else 0

    # ---------------- METRICS ----------------
    rhythm, sync_diff, sway, accel_var, jerk_var = compute_metrics(
        hip_x, hip_y, left_y, right_y, time_list
    )

    avg_angle = safe_mean(torso_angles)

    # ---------------- SCORES ----------------
    bal = balance_score(sway)
    rhy = rhythm_score(rhythm)
    syn = sync_score(sync_diff)
    acc = accuracy_score(cone_hits)
    flu = fluidity_score(accel_var, jerk_var)
    turn = turning_score(avg_angle)

    mabc = bal + rhy + syn + acc + flu + turn

    t_score = time_score(total_time)

    # ---------------- FINAL ----------------
    if mabc >= 7:
        final_score = t_score
    elif mabc >= 5:
        final_score = t_score - 0.5
    elif mabc >= 3:
        final_score = t_score - 1
    else:
        final_score = min(t_score, 2)

    final_category = (
        "Excellent" if final_score >= 4.5 else
        "Above Average" if final_score >= 3.5 else
        "Average" if final_score >= 2.5 else
        "Below Average" if final_score >= 1.5 else
        "Poor"
    )

    print("------ ILLINOIS AGILITY RESULT ------")
    print(f"Time: {total_time:.2f}s")
    print(f"Cone Hits: {len(cone_hits)-1}/4")
    print(f"Rhythm: {rhythm:.3f}")
    print(f"Sync: {sync_diff:.3f}")
    print(f"Sway: {sway:.3f}")
    print(f"Accel: {accel_var:.3f}, Jerk: {jerk_var:.3f}")
    print(f"Torso Angle: {avg_angle:.2f}")
    print(f"MABC Score: {mabc}")
    print(f"Final Score: {final_score}")
    print(f"Category: {final_category}")
    # print(zero_cone_flag)

    return final_score, final_category

In [19]:
path = "data/agility_test_2.mp4"
illinois_agility(path)

C:\Users\nayan\AppData\Local\Programs\Python\Python310\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


------ ILLINOIS AGILITY RESULT ------
Time: 20.89s
Cone Hits: 3/4
Rhythm: 0.294
Sync: 0.029
Sway: 0.266
Accel: 0.287, Jerk: 0.499
Torso Angle: 25.15
MABC Score: 3
Final Score: 0
Category: Poor


(0, 'Poor')